# Neural Networks from Scratch — MLP, Backpropagation and Gradient Descent

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print('NumPy version:', np.__version__)

## Exercise 1: Multi-Layer Perceptron (MLP) from Scratch

In [ ]:
# ── Activation functions ──────────────────────────────────────
def relu(z):
    """Rectified Linear Unit: max(0, z)"""
    return np.maximum(0, z)

def sigmoid(z):
    """Sigmoid: 1 / (1 + exp(-z))"""
    return 1 / (1 + np.exp(-z))

# ── Network architecture ──────────────────────────────────────
# Input layer  : 2 neurons
# Hidden layer : 3 neurons (ReLU)
# Output layer : 1 neuron  (Sigmoid)

# Weights and biases (fixed for reproducibility)
W1 = np.array([[0.2, -0.3, 0.5],    # shape (2, 3): 2 inputs → 3 hidden neurons
               [0.4,  0.1, -0.2]])
b1 = np.array([0.1, -0.1, 0.2])     # shape (3,)

W2 = np.array([[0.6],               # shape (3, 1): 3 hidden → 1 output
               [-0.4],
               [0.3]])
b2 = np.array([0.05])               # shape (1,)

print('Architecture:')
print(f'  W1 shape: {W1.shape}  (input → hidden)')
print(f'  b1 shape: {b1.shape}')
print(f'  W2 shape: {W2.shape}  (hidden → output)')
print(f'  b2 shape: {b2.shape}')

In [ ]:
def forward_propagation(x):
    """
    Forward pass for the 2-3-1 MLP.
    Returns: z1, a1 (hidden pre/post-activation), z2, a2 (output pre/post-activation)
    """
    # Hidden layer
    z1 = np.dot(x, W1) + b1          # (2,) @ (2,3) + (3,) → (3,)
    a1 = relu(z1)                     # ReLU activation

    # Output layer
    z2 = np.dot(a1, W2) + b2         # (3,) @ (3,1) + (1,) → (1,)
    a2 = sigmoid(z2)                  # Sigmoid activation

    return z1, a1, z2, a2

def predict(x):
    _, _, _, a2 = forward_propagation(x)
    return float(a2)

# ── Case 1: inputs [2, 3] ─────────────────────────────────────
x1 = np.array([2, 3])
z1_c1, a1_c1, z2_c1, a2_c1 = forward_propagation(x1)

print('=== Case 1: inputs = [2, 3] ===')
print(f'  Hidden pre-activation  z1 = {z1_c1.round(4)}')
print(f'  Hidden post-activation a1 = {a1_c1.round(4)}  (ReLU)')
print(f'  Output pre-activation  z2 = {z2_c1.round(4)}')
print(f'  Output prediction      a2 = {float(a2_c1):.6f}  (Sigmoid)')
print(f'  Binary prediction         = {1 if float(a2_c1) > 0.5 else 0}')
print()

# ── Case 2: inputs [1, 5] ─────────────────────────────────────
x2 = np.array([1, 5])
z1_c2, a1_c2, z2_c2, a2_c2 = forward_propagation(x2)

print('=== Case 2: inputs = [1, 5] ===')
print(f'  Hidden pre-activation  z1 = {z1_c2.round(4)}')
print(f'  Hidden post-activation a1 = {a1_c2.round(4)}  (ReLU)')
print(f'  Output pre-activation  z2 = {z2_c2.round(4)}')
print(f'  Output prediction      a2 = {float(a2_c2):.6f}  (Sigmoid)')
print(f'  Binary prediction         = {1 if float(a2_c2) > 0.5 else 0}')

In [ ]:
# ── Visualise the MLP architecture ───────────────────────────
from matplotlib.patches import Circle, FancyArrowPatch

fig, ax = plt.subplots(figsize=(12, 7))
ax.set_xlim(0, 12)
ax.set_ylim(0, 7)
ax.axis('off')
ax.set_facecolor('#F8F9FA')
fig.patch.set_facecolor('#F8F9FA')

layer_x = [2, 6, 10]
positions = {
    'input':  [5.5, 3.5],
    'hidden': [6.0, 4.5, 3.0],
    'output': [4.5]
}
colors = {'input':'#4C72B0', 'hidden':'#55A868', 'output':'#E84040'}

# Draw connections
for iy, y_src in enumerate(positions['input']):
    for hy, y_tgt in enumerate(positions['hidden']):
        ax.annotate('', xy=(layer_x[1]-0.4, y_tgt), xytext=(layer_x[0]+0.4, y_src),
                    arrowprops=dict(arrowstyle='->', color='#AAAAAA', lw=0.9))
for hy, y_src in enumerate(positions['hidden']):
    ax.annotate('', xy=(layer_x[2]-0.4, positions['output'][0]),
                xytext=(layer_x[1]+0.4, y_src),
                arrowprops=dict(arrowstyle='->', color='#AAAAAA', lw=0.9))

# Draw neurons
for i, (lx, ly, label, color) in enumerate([
    *[(layer_x[0], y, f'x{i+1}', colors['input']) for i, y in enumerate(positions['input'])],
    *[(layer_x[1], y, f'h{i+1}', colors['hidden']) for i, y in enumerate(positions['hidden'])],
    (layer_x[2], positions['output'][0], 'ŷ', colors['output'])
]):
    circ = Circle((lx, ly), 0.38, color=color, zorder=4, ec='white', lw=2)
    ax.add_patch(circ)
    ax.text(lx, ly, label, ha='center', va='center', color='white',
            fontsize=10, fontweight='bold', zorder=5)

# Layer titles
ax.text(layer_x[0], 1.8, 'Input Layer\n(2 neurons)', ha='center', fontsize=10,
        fontweight='bold', color=colors['input'],
        bbox=dict(boxstyle='round', facecolor='white', edgecolor=colors['input']))
ax.text(layer_x[1], 1.8, 'Hidden Layer\n(3 neurons, ReLU)', ha='center', fontsize=10,
        fontweight='bold', color=colors['hidden'],
        bbox=dict(boxstyle='round', facecolor='white', edgecolor=colors['hidden']))
ax.text(layer_x[2], 1.8, 'Output Layer\n(1 neuron, Sigmoid)', ha='center', fontsize=10,
        fontweight='bold', color=colors['output'],
        bbox=dict(boxstyle='round', facecolor='white', edgecolor=colors['output']))

ax.set_title('Multi-Layer Perceptron (MLP) — 2 → 3 → 1', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Exercise 2: Backpropagation with Gradient Descent

In [ ]:
# ── Given data ────────────────────────────────────────────────
x_bp      = np.array([6, 75])    # study hours=6, previous score=75
w_bp      = np.array([0.4, 0.6]) # weights
b_bp      = 5.0                  # bias
y_true_bp = 85.0                 # actual exam score
lr        = 0.01                 # learning rate

# ── Step 1: Forward pass — predict exam score ─────────────────
y_pred_bp = np.dot(x_bp, w_bp) + b_bp
error     = y_true_bp - y_pred_bp

print('=== Forward Propagation ===')
print(f'  z = ({x_bp[0]} × {w_bp[0]}) + ({x_bp[1]} × {w_bp[1]}) + {b_bp}')
print(f'  z = {x_bp[0]*w_bp[0]} + {x_bp[1]*w_bp[1]} + {b_bp}')
print(f'  Predicted score y_pred = {y_pred_bp:.2f}')
print(f'  Actual score    y_true = {y_true_bp:.2f}')
print(f'  Error (y_true - y_pred) = {error:.2f}')

In [ ]:
# ── Step 2: Compute MSE loss ──────────────────────────────────
loss = 0.5 * error ** 2

print('=== MSE Loss ===')
print(f'  Loss = 0.5 × (y_true - y_pred)² = 0.5 × ({error:.2f})² = {loss:.4f}')

In [ ]:
# ── Step 3: Compute gradients and update weights ──────────────
# ∂Loss/∂w = -(y_true - y_pred) × x
# ∂Loss/∂b = -(y_true - y_pred)
grad_w = -error * x_bp
grad_b = -error

w_new = w_bp - lr * grad_w
b_new = b_bp - lr * grad_b

print('=== Gradients ===')
print(f'  ∂Loss/∂w1 = -({error:.2f}) × {x_bp[0]} = {grad_w[0]:.4f}')
print(f'  ∂Loss/∂w2 = -({error:.2f}) × {x_bp[1]} = {grad_w[1]:.4f}')
print(f'  ∂Loss/∂b  = -({error:.2f})              = {grad_b:.4f}')
print()
print('=== Weight Update (lr = 0.01) ===')
print(f'  w1: {w_bp[0]:.4f} - {lr} × ({grad_w[0]:.4f}) = {w_new[0]:.6f}')
print(f'  w2: {w_bp[1]:.4f} - {lr} × ({grad_w[1]:.4f}) = {w_new[1]:.6f}')
print(f'  b : {b_bp:.4f}  - {lr} × ({grad_b:.4f})   = {b_new:.6f}')
print()
print('=== Interpretation ===')
print(f'  The model overpredicted by {-error:.2f} points.')
print(f'  Gradient descent reduces w1 and w2 to lower future predictions.')
print(f'  The bias also decreases slightly, shifting all predictions downward.')

In [ ]:
# ── Verify: new prediction is closer to y_true ────────────────
y_pred_new = np.dot(x_bp, w_new) + b_new
print(f'  New prediction after 1 step : {y_pred_new:.4f}')
print(f'  Previous error              : {abs(error):.4f}')
print(f'  New error                   : {abs(y_true_bp - y_pred_new):.4f}')
print(f'  Improvement                 : {abs(error) - abs(y_true_bp - y_pred_new):.4f}')

## Exercise 3: Comparing Activation Functions

In [ ]:
# ── Implement the three activation functions ──────────────────
def step_function(z):
    """Step function: 1 if z > 0, else 0."""
    return np.where(z > 0, 1, 0)

def sigmoid_fn(z):
    """Sigmoid: smooth S-curve mapping z to (0, 1)."""
    return 1 / (1 + np.exp(-z))

def relu_fn(z):
    """ReLU: max(0, z)."""
    return np.maximum(0, z)

# Test values
z_test = np.array([-2, -1, 0, 1, 2])
print(f'{"z":>5}  {"Step":>6}  {"Sigmoid":>8}  {"ReLU":>6}')
print('-' * 32)
for z in z_test:
    print(f'{z:>5}  {step_function(z):>6}  {sigmoid_fn(z):>8.4f}  {relu_fn(z):>6}')

In [ ]:
# ── Visualise all three ───────────────────────────────────────
z_range = np.linspace(-5, 5, 400)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, fn, name, color, desc in [
    (axes[0], step_function, 'Step Function',   '#C44E52',
     'Binary output (0 or 1)\nNon-differentiable at z=0'),
    (axes[1], sigmoid_fn,    'Sigmoid Function', '#4C72B0',
     'Smooth S-curve: output ∈ (0,1)\nGradient vanishes for large |z|'),
    (axes[2], relu_fn,       'ReLU Function',    '#55A868',
     'Linear for z>0, zero otherwise\nNo vanishing gradient (positive side)'),
]:
    ax.plot(z_range, fn(z_range), color=color, linewidth=2.5)
    ax.axhline(0, color='black', linewidth=0.7, alpha=0.4)
    ax.axvline(0, color='black', linewidth=0.7, alpha=0.4)
    ax.set_title(name, fontsize=13, fontweight='bold', color=color)
    ax.set_xlabel('z (input)')
    ax.set_ylabel('f(z) (output)')
    ax.text(0.03, 0.97, desc, transform=ax.transAxes, fontsize=8.5,
            va='top', ha='left', style='italic',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    ax.grid(True, alpha=0.3)

plt.suptitle('Activation Functions Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Derivative comparison ─────────────────────────────────────
def sigmoid_deriv(z):
    s = sigmoid_fn(z)
    return s * (1 - s)

def relu_deriv(z):
    return np.where(z > 0, 1.0, 0.0)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(z_range, sigmoid_deriv(z_range), color='#4C72B0', linewidth=2, label="Sigmoid' (max=0.25)")
ax.plot(z_range, relu_deriv(z_range),    color='#55A868', linewidth=2, label="ReLU' (0 or 1)")
ax.set_title('Derivatives of Activation Functions', fontsize=13, fontweight='bold')
ax.set_xlabel('z')
ax.set_ylabel("f'(z)")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Answers to the Questions

**1. Which function gives only binary outputs?**  
The **Step function** — it outputs exactly 0 or 1, with a hard discontinuity at z = 0. It cannot produce any intermediate probability.

**2. Which function smoothly transitions between values?**  
The **Sigmoid function** — it produces a smooth S-curve, transitioning continuously from ~0 (for large negative z) to ~1 (for large positive z), passing through 0.5 at z = 0.

**3. Which function sets negative values to zero but keeps positive values unchanged?**  
**ReLU** — `f(z) = max(0, z)`. Negative inputs are clipped to 0; positive inputs pass through unchanged.

**4. Why is ReLU commonly used in deep learning?**  
ReLU avoids the **vanishing gradient problem**: its derivative is 1 for all positive z, meaning gradients flow back through many layers without shrinking. This makes deep networks trainable. It is also computationally trivial (a single `max` comparison), enabling fast training on large architectures.

**5. Why might Sigmoid be a good choice for binary classification output layers?**  
Sigmoid naturally maps any real number to the interval (0, 1), making the output directly interpretable as a **probability**. For binary classification, a threshold of 0.5 cleanly separates positive from negative predictions.

**6. Weaknesses of the Step function:**  
- **Non-differentiable** at z = 0 and has zero gradient everywhere else → backpropagation cannot update weights.  
- **No probabilistic output** — only hard 0/1 decisions, which makes the model unable to express confidence.  
- Practically useless for gradient-based training; only useful for theoretical perceptron analysis.

## Exercise 4: Forward Propagation in a Deep Neural Network

In [ ]:
# ── Given data ────────────────────────────────────────────────
x_house = np.array([2000, 3])     # sq ft, bedrooms

# Layer 1
W_L1 = np.array([0.5, 0.7])
b_L1 = 10_000.0

# Layer 2
W_L2 = np.array([0.6, 0.8])
b_L2 = 20_000.0

# Output layer
W_out = 1.2
b_out = 30_000.0

def relu_scalar(z):
    return max(0, z)

# ── Layer 1 ───────────────────────────────────────────────────
z_L1 = np.dot(x_house, W_L1) + b_L1
a_L1 = relu_scalar(z_L1)

print('=== Layer 1 ===')
print(f'  z_L1 = ({x_house[0]}×{W_L1[0]}) + ({x_house[1]}×{W_L1[1]}) + {b_L1:,.0f}')
print(f'       = {x_house[0]*W_L1[0]:,.0f} + {x_house[1]*W_L1[1]:.1f} + {b_L1:,.0f}')
print(f'       = {z_L1:,.1f}')
print(f'  a_L1 = ReLU({z_L1:,.1f}) = {a_L1:,.1f}')
print()

# ── Layer 2 ───────────────────────────────────────────────────
# Layer 2 takes a_L1 as a single value; using same-sized weight vector
z_L2 = a_L1 * W_L2[0] + a_L1 * W_L2[1] + b_L2   # both weights applied to a_L1
a_L2 = relu_scalar(z_L2)

print('=== Layer 2 ===')
print(f'  z_L2 = ({a_L1:,.1f}×{W_L2[0]}) + ({a_L1:,.1f}×{W_L2[1]}) + {b_L2:,.0f}')
print(f'       = {a_L1*W_L2[0]:,.1f} + {a_L1*W_L2[1]:,.1f} + {b_L2:,.0f}')
print(f'       = {z_L2:,.1f}')
print(f'  a_L2 = ReLU({z_L2:,.1f}) = {a_L2:,.1f}')
print()

# ── Output layer ──────────────────────────────────────────────
z_out  = a_L2 * W_out + b_out
price  = relu_scalar(z_out)   # ReLU ensures price ≥ 0

print('=== Output Layer ===')
print(f'  z_out = {a_L2:,.1f} × {W_out} + {b_out:,.0f}')
print(f'        = {a_L2*W_out:,.1f} + {b_out:,.0f}')
print(f'        = {z_out:,.1f}')
print(f'  ReLU(z_out) = {price:,.1f}')
print()
print(f'Predicted House Price: ${price:,.0f}')

In [ ]:
# ── Visualise forward propagation flow ────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
ax.set_xlim(0, 14)
ax.set_ylim(0, 5)
ax.axis('off')
ax.set_facecolor('#F8F9FA')
fig.patch.set_facecolor('#F8F9FA')

stages = [
    (1.5, 2.5, 'Inputs\n2000 sqft\n3 beds',          '#4C72B0'),
    (4.5, 2.5, f'Layer 1\nz={z_L1:,.0f}\na={a_L1:,.0f}', '#55A868'),
    (7.5, 2.5, f'Layer 2\nz={z_L2:,.0f}\na={a_L2:,.0f}', '#DD8452'),
    (10.5, 2.5, f'Output\n${price:,.0f}',              '#E84040'),
]

for x, y, text, color in stages:
    ax.add_patch(plt.FancyBboxPatch((x-1.2, y-1.0), 2.4, 2.0,
                                    boxstyle='round,pad=0.1',
                                    facecolor=color, edgecolor='white',
                                    linewidth=2, alpha=0.9))
    ax.text(x, y, text, ha='center', va='center', color='white',
            fontsize=9, fontweight='bold')

for i in range(len(stages)-1):
    x1 = stages[i][0] + 1.2
    x2 = stages[i+1][0] - 1.2
    y  = 2.5
    ax.annotate('', xy=(x2, y), xytext=(x1, y),
                arrowprops=dict(arrowstyle='->', color='#555', lw=2))
    mid = (x1+x2)/2
    labels = ['W_L1, b_L1\n+ ReLU', 'W_L2, b_L2\n+ ReLU', 'W_out, b_out\n+ ReLU']
    ax.text(mid, 3.8, labels[i], ha='center', fontsize=8, color='#444', style='italic')

ax.set_title('Forward Propagation — 3-Layer House Price Network', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Interpretation

The three-layer network computes progressively refined feature transformations of the raw input (2000 sq ft, 3 bedrooms):
- **Layer 1** applies a linear combination with weights (0.5 and 0.7) plus a base offset of $10,000, then ReLU clips any negative result to zero.
- **Layer 2** further scales Layer 1's output and adds a second base price component of $20,000, again clipped by ReLU.
- The **output layer** scales and shifts the final representation to produce the predicted house price.

The ReLU function at each stage ensures the predicted price can never be negative, which is physically meaningful.

## Exercise 5: Training a Neural Network with Forward and Backward Propagation

In [ ]:
# ── Training loop: 200 iterations ────────────────────────────
x_train_5  = np.array([4, 80])    # study hours=4, previous score=80
y_true_5   = 85.0                 # actual exam score
w_5        = np.array([0.6, 0.3]) # initial weights
b_5        = 10.0                 # initial bias
lr_5       = 0.01
N_EPOCHS   = 200

history_5 = {'loss': [], 'pred': [], 'w0': [], 'w1': [], 'b': []}

for epoch in range(N_EPOCHS):
    # Forward propagation
    y_pred_5 = np.dot(x_train_5, w_5) + b_5

    # MSE loss
    loss_5   = 0.5 * (y_true_5 - y_pred_5) ** 2

    # Gradients (backpropagation)
    grad_w = -(y_true_5 - y_pred_5) * x_train_5
    grad_b = -(y_true_5 - y_pred_5)

    # Update weights and bias
    w_5 = w_5 - lr_5 * grad_w
    b_5 = b_5 - lr_5 * grad_b

    history_5['loss'].append(loss_5)
    history_5['pred'].append(y_pred_5)
    history_5['w0'].append(w_5[0])
    history_5['w1'].append(w_5[1])
    history_5['b'].append(b_5)

print('=== Initial State (before training) ===')
initial_pred = np.dot(x_train_5, np.array([0.6, 0.3])) + 10.0
print(f'  Initial prediction : {initial_pred:.4f}')
print(f'  Initial loss       : {0.5*(y_true_5 - initial_pred)**2:.4f}')
print()
print('=== After 1 Training Iteration ===')
print(f'  Prediction  : {history_5["pred"][0]:.4f}')
print(f'  Loss        : {history_5["loss"][0]:.4f}')
print(f'  w0 (updated): {history_5["w0"][0]:.6f}')
print(f'  w1 (updated): {history_5["w1"][0]:.6f}')
print(f'  b  (updated): {history_5["b"][0]:.6f}')
print()
print(f'=== After {N_EPOCHS} Training Iterations ===')
print(f'  Final prediction : {history_5["pred"][-1]:.4f}')
print(f'  Final loss       : {history_5["loss"][-1]:.6f}')
print(f'  Final w0         : {history_5["w0"][-1]:.6f}')
print(f'  Final w1         : {history_5["w1"][-1]:.6f}')
print(f'  Final b          : {history_5["b"][-1]:.6f}')

In [ ]:
# ── Visualise training convergence ────────────────────────────
epochs_ax = range(1, N_EPOCHS + 1)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Loss
axes[0].plot(epochs_ax, history_5['loss'], color='crimson', linewidth=2)
axes[0].set_title('Loss over Training Iterations', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('MSE Loss')
axes[0].set_yscale('log')
axes[0].grid(True, alpha=0.3)

# Prediction convergence
axes[1].plot(epochs_ax, history_5['pred'], color='steelblue', linewidth=2)
axes[1].axhline(y_true_5, color='crimson', linestyle='--', linewidth=1.8,
                label=f'Target ({y_true_5})')
axes[1].set_title('Prediction Convergence', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Predicted Score')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

# Weight evolution
axes[2].plot(epochs_ax, history_5['w0'], linewidth=2, label='w₀ (study hours)')
axes[2].plot(epochs_ax, history_5['w1'], linewidth=2, label='w₁ (prev score)')
axes[2].plot(epochs_ax, history_5['b'],  linewidth=2, linestyle='--', label='bias')
axes[2].set_title('Weight & Bias Evolution', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Iteration')
axes[2].set_ylabel('Value')
axes[2].legend(fontsize=9)
axes[2].grid(True, alpha=0.3)

plt.suptitle('Training Loop — Gradient Descent Convergence', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Effect of learning rate on convergence ────────────────────
def train_lr(lr, n_epochs=200):
    w = np.array([0.6, 0.3])
    b = 10.0
    losses = []
    for _ in range(n_epochs):
        yp   = np.dot(x_train_5, w) + b
        err  = y_true_5 - yp
        losses.append(0.5 * err**2)
        w   -= lr * (-err * x_train_5)
        b   -= lr * (-err)
    return losses

lrs = [0.001, 0.005, 0.01, 0.05]
fig, ax = plt.subplots(figsize=(10, 5))
for lr_val in lrs:
    ax.plot(train_lr(lr_val), linewidth=2, label=f'lr = {lr_val}')
ax.set_yscale('log')
ax.set_title('Effect of Learning Rate on Convergence', fontsize=13, fontweight='bold')
ax.set_xlabel('Iteration')
ax.set_ylabel('Loss (log scale)')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

| Exercise | Key Concept | Implementation |
|---|---|---|
| 1 | Multi-Layer Perceptron | 2→3→1 network with ReLU hidden + Sigmoid output, forward propagation from scratch |
| 2 | Backpropagation | MSE loss, analytical gradients, one-step weight update via gradient descent |
| 3 | Activation Functions | Step (binary), Sigmoid (smooth), ReLU (sparse), with derivative comparison |
| 4 | Deep Forward Propagation | 3-layer house price network; layer-by-layer computation with ReLU |
| 5 | Full Training Loop | 200-iteration gradient descent; loss, prediction, weight evolution visualised |

**Key takeaway:** Every component of a neural network — architecture, activation function, loss function, learning rate — interacts to determine how quickly and accurately the model learns. Starting from scratch with NumPy makes these interactions transparent before moving to high-level frameworks like Keras.